In [ ]:
# ============================================================
# STEP 4 - CLUSTERING FROM DGI NODE EMBEDDINGS
# Original vs synthetic DGI embeddings
# + comparison of post-training vs training-time embeddings
# KMeans / GMM / Spectral + interpretation
# ============================================================

# ============================================================
# CELL 1 - Imports
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
import seaborn as sns

from sklearn.cluster import KMeans, SpectralClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import (
    silhouette_score,
    calinski_harabasz_score,
    davies_bouldin_score,
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
)
from sklearn.manifold import TSNE
from sklearn.feature_selection import mutual_info_classif
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text

# ============================================================
# ADDITIONAL EXPERIMENT
# ============================================================
# This Step 4 compares:
#
# - Multiple Encoders: current independent-encoder embeddings
# - One Encoder:
#       (a) post-training re-encoded embeddings
#       (b) embeddings produced directly during training
# - Batch:
#       (a) post-training re-encoded embeddings
#       (b) embeddings produced directly during training
#
# The clustering pipeline is identical for the two variants.
# ============================================================


In [ ]:
# ============================================================
# CELL 2 - Load embeddings and feature files
# ============================================================

def load_embeddings(embeddings_path):
    return np.load(embeddings_path)


def load_feature_names(featnames_path):
    featnames_raw = pd.read_csv(
        featnames_path,
        sep=r"\s+",
        header=None,
        engine="python"
    )

    if featnames_raw.shape[1] > 2:
        feature_names = featnames_raw.iloc[:, 1:].astype(str).agg("_".join, axis=1)
    else:
        feature_names = featnames_raw.iloc[:, 1].astype(str)

    return feature_names.tolist()


def load_features_for_synthetic_graphs(features_path, featnames_path=None):
    features_df = pd.read_csv(features_path, index_col=0)

    features_df = features_df.reset_index(drop=True)
    features_df.index = range(1, len(features_df) + 1)

    if featnames_path is not None:
        feature_names = load_feature_names(featnames_path)

        if len(feature_names) == features_df.shape[1]:
            features_df.columns = feature_names
        else:
            print("Warning: number of feature names does not match feature columns.")
            print("Feature columns:", features_df.shape[1])
            print("Feature names:", len(feature_names))

    return features_df


def load_ard_features(ard_path):
    ard_df = pd.read_csv(ard_path, index_col=0)

    ard_df = ard_df.reset_index(drop=True)
    ard_df.index = range(1, len(ard_df) + 1)

    return ard_df

In [ ]:
DISPLAY_NAMES = {
    "original_all_features": "Original Graph",
    "original_circles_only": "Original Graph",
    "original_graph_structure_identity_only": "Original Graph",

    "original_TRAINING_all_features": "Original Graph - Training Embeddings",
    "original_TRAINING_circles_only": "Original Graph - Training Embeddings",
    "original_TRAINING_graph_structure_identity_only": "Original Graph - Training Embeddings",

    "method1_independent_mean_all_features": "Multiple Encoders",
    "method1_independent_mean_circles_only": "Multiple Encoders",
    "method1_independent_mean_graph_structure_identity_only": "Multiple Encoders",

    "method2_shared_sequential_mean_all_features": "One Encoder",
    "method2_shared_sequential_mean_circles_only": "One Encoder",
    "method2_shared_sequential_mean_graph_structure_identity_only": "One Encoder",

    "method3_batch_mean_all_features": "Batch",
    "method3_batch_mean_circles_only": "Batch",
    "method3_batch_mean_graph_structure_identity_only": "Batch",

    "method2_shared_sequential_TRAINING_MEAN_all_features": "One Encoder - Training Embeddings",
    "method2_shared_sequential_TRAINING_MEAN_circles_only": "One Encoder - Training Embeddings",
    "method2_shared_sequential_TRAINING_MEAN_graph_structure_identity_only": "One Encoder - Training Embeddings",

    "method3_batch_TRAINING_MEAN_all_features": "Batch - Training Embeddings",
    "method3_batch_TRAINING_MEAN_circles_only": "Batch - Training Embeddings",
    "method3_batch_TRAINING_MEAN_graph_structure_identity_only": "Batch - Training Embeddings",
}

In [ ]:
# ============================================================
# CELL 3 - Load membership matrix and add circleOverlap
# ============================================================

def load_membership_for_synthetic_order(membership_path):
    membership = pd.read_csv(membership_path, index_col=0)

    membership = membership.reset_index(drop=True)
    membership.index = range(1, len(membership) + 1)

    return membership


def add_circle_overlap_column(membership_matrix):
    membership_extended = membership_matrix.copy()

    overlap_nodes = membership_extended[
        membership_extended.sum(axis=1) > 1
    ].index

    membership_extended["circleOverlap"] = 0

    for node in overlap_nodes:
        membership_extended.loc[
            node,
            membership_extended.columns != "circleOverlap"
        ] = 0
        membership_extended.loc[node, "circleOverlap"] = 1

    print("Number of overlap nodes:", len(overlap_nodes))

    return membership_extended

In [ ]:
# ============================================================
# CELL 4 - Clustering diagnostics from k=2 to k=8
# ============================================================

def compute_clustering_diagnostics(
    embeddings,
    k_min=2,
    k_max=8,
    random_state=123
):
    results = []

    for k in range(k_min, k_max + 1):

        kmeans = KMeans(
            n_clusters=k,
            random_state=random_state,
            n_init=10
        )

        labels_kmeans = kmeans.fit_predict(embeddings)

        results.append({
            "method": "kmeans",
            "k": k,
            "silhouette": silhouette_score(embeddings, labels_kmeans),
            "calinski_harabasz": calinski_harabasz_score(embeddings, labels_kmeans),
            "davies_bouldin": davies_bouldin_score(embeddings, labels_kmeans),
        })

        gmm = GaussianMixture(
            n_components=k,
            covariance_type="diag",
            reg_covar=1e-4,
            n_init=5,
            random_state=random_state
        )

        labels_gmm = gmm.fit_predict(embeddings)

        results.append({
            "method": "gmm",
            "k": k,
            "silhouette": silhouette_score(embeddings, labels_gmm),
            "calinski_harabasz": calinski_harabasz_score(embeddings, labels_gmm),
            "davies_bouldin": davies_bouldin_score(embeddings, labels_gmm),
        })

        spectral = SpectralClustering(
            n_clusters=k,
            affinity="rbf",
            random_state=random_state
        )

        labels_spectral = spectral.fit_predict(embeddings)

        results.append({
            "method": "spectral",
            "k": k,
            "silhouette": silhouette_score(embeddings, labels_spectral),
            "calinski_harabasz": calinski_harabasz_score(embeddings, labels_spectral),
            "davies_bouldin": davies_bouldin_score(embeddings, labels_spectral),
        })

    return pd.DataFrame(results)


def split_diagnostics_by_method(diagnostics_df, output_folder, dataset_name):
    diagnostics_by_method = {}

    for method in diagnostics_df["method"].unique():
        df_method = diagnostics_df[
            diagnostics_df["method"] == method
        ].copy()

        diagnostics_by_method[method] = df_method

        df_method.to_csv(
            os.path.join(
                output_folder,
                f"{dataset_name}_diagnostics_{method}_k2_k8.csv"
            ),
            index=False
        )

        print(f"\n=== {dataset_name} | {method.upper()} diagnostics ===")
        print(df_method)

    return diagnostics_by_method

In [ ]:
# ============================================================
# CELL 3 - Load membership matrix and add circleOverlap
# ============================================================

def load_membership_for_synthetic_order(membership_path):
    membership = pd.read_csv(membership_path, index_col=0)

    membership = membership.reset_index(drop=True)
    membership.index = range(1, len(membership) + 1)

    return membership


def add_circle_overlap_column(membership_matrix):
    membership_extended = membership_matrix.copy()

    overlap_nodes = membership_extended[
        membership_extended.sum(axis=1) > 1
    ].index

    membership_extended["circleOverlap"] = 0

    for node in overlap_nodes:
        membership_extended.loc[
            node,
            membership_extended.columns != "circleOverlap"
        ] = 0
        membership_extended.loc[node, "circleOverlap"] = 1

    print("Number of overlap nodes:", len(overlap_nodes))

    return membership_extended

In [ ]:
# ============================================================
# CELL 4 - Clustering diagnostics from k=2 to k=8
# ============================================================

def compute_internal_clustering_metrics(embeddings, labels):
    n_samples = len(labels)
    n_clusters_found = len(np.unique(labels))

    if n_clusters_found < 2 or n_clusters_found >= n_samples:
        return {
            "silhouette": np.nan,
            "calinski_harabasz": np.nan,
            "davies_bouldin": np.nan,
            "n_clusters_found": n_clusters_found,
            "valid_internal_metrics": False,
        }

    return {
        "silhouette": silhouette_score(embeddings, labels),
        "calinski_harabasz": calinski_harabasz_score(
            embeddings,
            labels
        ),
        "davies_bouldin": davies_bouldin_score(
            embeddings,
            labels
        ),
        "n_clusters_found": n_clusters_found,
        "valid_internal_metrics": True,
    }


def fit_clustering_method_safely(
    method_name,
    embeddings,
    k,
    random_state
):
    try:
        if method_name == "kmeans":
            model = KMeans(
                n_clusters=k,
                random_state=random_state,
                n_init=10
            )

        elif method_name == "gmm":
            model = GaussianMixture(
                n_components=k,
                covariance_type="diag",
                reg_covar=1e-4,
                n_init=5,
                random_state=random_state
            )

        elif method_name == "spectral":
            model = SpectralClustering(
                n_clusters=k,
                affinity="rbf",
                random_state=random_state
            )

        else:
            raise ValueError(f"Unknown method: {method_name}")

        labels = model.fit_predict(embeddings)

        metrics = compute_internal_clustering_metrics(
            embeddings,
            labels
        )

        metrics["method"] = method_name
        metrics["k"] = k
        metrics["fit_status"] = "ok"

        if not metrics["valid_internal_metrics"]:
            metrics["fit_status"] = (
                "collapsed_or_invalid_number_of_clusters"
            )

        return metrics

    except Exception as error:
        return {
            "method": method_name,
            "k": k,
            "silhouette": np.nan,
            "calinski_harabasz": np.nan,
            "davies_bouldin": np.nan,
            "n_clusters_found": np.nan,
            "valid_internal_metrics": False,
            "fit_status": f"error: {type(error).__name__}",
        }


def compute_clustering_diagnostics(
    embeddings,
    k_min=2,
    k_max=8,
    random_state=123
):
    results = []

    for k in range(k_min, k_max + 1):
        for method_name in ["kmeans", "gmm", "spectral"]:
            results.append(
                fit_clustering_method_safely(
                    method_name=method_name,
                    embeddings=embeddings,
                    k=k,
                    random_state=random_state
                )
            )

    return pd.DataFrame(results)


def split_diagnostics_by_method(diagnostics_df, output_folder, dataset_name):
    diagnostics_by_method = {}

    for method in diagnostics_df["method"].unique():
        df_method = diagnostics_df[
            diagnostics_df["method"] == method
        ].copy()

        diagnostics_by_method[method] = df_method

        df_method.to_csv(
            os.path.join(
                output_folder,
                f"{dataset_name}_diagnostics_{method}_k2_k8.csv"
            ),
            index=False
        )

        print(f"\n=== {dataset_name} | {method.upper()} diagnostics ===")
        print(df_method)

    return diagnostics_by_method

In [ ]:
# ============================================================
# CELL 5 - Clustering models
# ============================================================

def evaluate_clustering(embeddings, labels):
    n_clusters_found = len(np.unique(labels))
    n_samples = len(labels)

    if n_clusters_found < 2 or n_clusters_found >= n_samples:
        return {
            "silhouette": np.nan,
            "calinski_harabasz": np.nan,
            "davies_bouldin": np.nan,
            "n_clusters": n_clusters_found,
            "valid_internal_metrics": False,
            "fit_status": "collapsed_or_invalid_number_of_clusters",
        }

    return {
        "silhouette": silhouette_score(embeddings, labels),
        "calinski_harabasz": calinski_harabasz_score(
            embeddings,
            labels
        ),
        "davies_bouldin": davies_bouldin_score(
            embeddings,
            labels
        ),
        "n_clusters": n_clusters_found,
        "valid_internal_metrics": True,
        "fit_status": "ok",
    }


def run_kmeans(embeddings, n_clusters, random_state=123):
    model = KMeans(
        n_clusters=n_clusters,
        random_state=random_state,
        n_init=10
    )

    labels = model.fit_predict(embeddings)
    metrics = evaluate_clustering(embeddings, labels)

    return labels, metrics, model, model.cluster_centers_


def run_gmm(embeddings, n_clusters, random_state=123):
    model = GaussianMixture(
        n_components=n_clusters,
        covariance_type="diag",
        reg_covar=1e-4,
        n_init=5,
        random_state=random_state
    )

    labels = model.fit_predict(embeddings)
    metrics = evaluate_clustering(embeddings, labels)

    return labels, metrics, model


def run_spectral(embeddings, n_clusters, random_state=123):
    model = SpectralClustering(
        n_clusters=n_clusters,
        affinity="rbf",
        random_state=random_state
    )

    labels = model.fit_predict(embeddings)
    metrics = evaluate_clustering(embeddings, labels)

    return labels, metrics, model


def make_failed_clustering_result(error_message):
    return {
        "labels": None,
        "metrics": {
            "silhouette": np.nan,
            "calinski_harabasz": np.nan,
            "davies_bouldin": np.nan,
            "n_clusters": np.nan,
            "valid_internal_metrics": False,
            "fit_status": error_message,
        },
        "model": None,
    }


def compare_clustering_methods(embeddings, n_clusters, random_state=123):
    results = {}

    try:
        labels, metrics, model, centroides = run_kmeans(
            embeddings,
            n_clusters,
            random_state
        )

        results["kmeans"] = {
            "labels": labels,
            "metrics": metrics,
            "model": model,
            "centroides": centroides,
        }

    except Exception as error:
        results["kmeans"] = make_failed_clustering_result(
            f"error: {type(error).__name__}"
        )
        results["kmeans"]["centroides"] = None

    try:
        labels, metrics, model = run_gmm(
            embeddings,
            n_clusters,
            random_state
        )

        results["gmm"] = {
            "labels": labels,
            "metrics": metrics,
            "model": model,
        }

    except Exception as error:
        results["gmm"] = make_failed_clustering_result(
            f"error: {type(error).__name__}"
        )

    try:
        labels, metrics, model = run_spectral(
            embeddings,
            n_clusters,
            random_state
        )

        results["spectral"] = {
            "labels": labels,
            "metrics": metrics,
            "model": model,
        }

    except Exception as error:
        results["spectral"] = make_failed_clustering_result(
            f"error: {type(error).__name__}"
        )

    metrics_df = pd.DataFrame({
        method: result["metrics"]
        for method, result in results.items()
    }).T

    return results, metrics_df

In [ ]:
# ============================================================
# CELL 6 - Visualization
# ============================================================

def compute_tsne(embeddings, random_state=123):
    tsne = TSNE(
        n_components=2,
        random_state=random_state,
        init="pca",
        learning_rate="auto"
    )

    emb_2d = tsne.fit_transform(embeddings)

    return emb_2d


def plot_tsne_clusters(emb_2d, labels, title="DGI Embeddings - t-SNE"):
    plt.figure(figsize=(8, 6))
    plt.scatter(emb_2d[:, 0], emb_2d[:, 1], c=labels, cmap="tab10")
    plt.title(title)
    plt.xlabel("t-SNE 1")
    plt.ylabel("t-SNE 2")
    plt.tight_layout()
    plt.show()


def plot_cluster_sizes(labels):
    cluster_counts = pd.Series(labels).value_counts().sort_index()

    plt.figure(figsize=(8, 5))
    sns.barplot(x=cluster_counts.index, y=cluster_counts.values, palette="Paired")
    plt.xlabel("Cluster ID")
    plt.ylabel("Number of Nodes")
    plt.title("Cluster Sizes")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

    return cluster_counts


def plot_feature_sparsity(features_df):
    feature_means = features_df.mean(axis=0).sort_values(
        ascending=False
    )

    plt.figure(figsize=(10, 5))
    feature_means.plot(kind="bar")

    plt.title("Feature Prevalence")
    plt.xlabel("Feature")
    plt.ylabel("Mean value")
    plt.tight_layout()
    plt.show()

    return feature_means


In [ ]:
# ============================================================
# CELL 7 - Save cluster labels
# ============================================================

def save_cluster_labels(labels, output_path):
    df = pd.DataFrame({
        "node_position": np.arange(1, len(labels) + 1),
        "cluster": labels,
    })

    df.to_csv(output_path, index=False)

    return df

In [ ]:
# ============================================================
# CELL 8 - Feature interpretation by cluster
# ============================================================

def compute_cluster_feature_profiles(features_df, labels):
    df = features_df.copy()
    df["cluster"] = labels

    cluster_feature_proportions = df.groupby("cluster").mean()

    return cluster_feature_proportions


def plot_feature_profile_heatmap(cluster_feature_proportions):
    plt.figure(figsize=(12, 8))
    sns.heatmap(
        cluster_feature_proportions.T,
        cmap="Blues",
        annot=True,
        fmt=".2f"
    )

    plt.title("Mean Feature Value Per Cluster")
    plt.xlabel("Cluster")
    plt.ylabel("Feature")
    plt.tight_layout()
    plt.show()


def print_features_above_threshold(cluster_feature_proportions, threshold=0.3):
    mask = cluster_feature_proportions > threshold
    features_above_threshold = mask.any(axis=0)

    features_over_threshold = cluster_feature_proportions.loc[:, features_above_threshold]

    sorted_features = features_over_threshold.loc[
        :,
        features_over_threshold.max() > threshold
    ]

    print(f"Features with > {int(threshold * 100)}% prevalence/value in at least one cluster:")
    print(sorted_features)

    return sorted_features


def compute_mutual_information_features(features_df, labels):
    X = features_df.to_numpy()
    y = labels

    mi = mutual_info_classif(
        X,
        y,
        discrete_features=True,
        random_state=123
    )

    mi_scores = pd.Series(mi, index=features_df.columns).sort_values(ascending=False)

    return mi_scores


def plot_top_features_per_cluster(cluster_feature_proportions, threshold=0.4):
    num_clusters = cluster_feature_proportions.shape[0]

    fig, axes = plt.subplots(
        num_clusters,
        1,
        figsize=(12, 5 * num_clusters),
        constrained_layout=True
    )

    if num_clusters == 1:
        axes = [axes]

    for i, cluster_id in enumerate(cluster_feature_proportions.index):
        prevalence = cluster_feature_proportions.loc[cluster_id].sort_values(ascending=False)
        top_prevalence = prevalence[prevalence > threshold]

        axes[i].bar(
            top_prevalence.index,
            top_prevalence.values,
            color="steelblue"
        )

        axes[i].set_title(
            f"Cluster {cluster_id} - Top Features > {threshold}"
        )

        axes[i].set_ylabel("Mean value")
        axes[i].tick_params(axis="x", rotation=45)

    plt.suptitle("Feature Values per Cluster", fontsize=16)
    plt.show()

In [ ]:
# ============================================================
# CELL 9 - CLAMP-style cluster prototypes
# ============================================================

def compute_clamp_feature_prototypes(
    features_df,
    labels,
    lower_q=0.10,
    upper_q=0.90,
    top_n=15
):
    df = features_df.copy()
    df["cluster"] = labels

    global_mean = features_df.mean(axis=0)
    global_std = features_df.std(axis=0)

    prototypes = {}

    for cluster_id, group in df.groupby("cluster"):
        X_cluster = group.drop(columns="cluster")

        cluster_mean = X_cluster.mean(axis=0)
        q_low = X_cluster.quantile(lower_q, axis=0)
        q_high = X_cluster.quantile(upper_q, axis=0)

        proto = pd.DataFrame({
            "cluster": cluster_id,
            "feature": features_df.columns,
            "cluster_prevalence": cluster_mean.values,
            "global_prevalence": global_mean.values,
            "prevalence_diff": (cluster_mean - global_mean).values,
            "abs_prevalence_diff": (cluster_mean - global_mean).abs().values,
            "lift": (cluster_mean / global_mean.replace(0, np.nan)).values,
            f"q{int(lower_q * 100)}": q_low.values,
            f"q{int(upper_q * 100)}": q_high.values,
            "global_std": global_std.values,
        })

        proto = proto.sort_values(
            "abs_prevalence_diff",
            ascending=False
        ).head(top_n)

        prototypes[cluster_id] = proto

    prototypes_df = pd.concat(prototypes.values(), axis=0).reset_index(drop=True)

    return prototypes_df


def plot_clamp_feature_prototypes(
    prototypes_df,
    top_n=10,
    value_col="cluster_prevalence"
):
    cluster_ids = sorted(prototypes_df["cluster"].unique())

    for cluster_id in cluster_ids:
        cluster_proto = prototypes_df[
            prototypes_df["cluster"] == cluster_id
        ].head(top_n).copy()

        cluster_proto = cluster_proto.sort_values(value_col, ascending=True)

        plt.figure(figsize=(10, max(4, 0.35 * len(cluster_proto))))

        plt.barh(
            cluster_proto["feature"],
            cluster_proto[value_col],
            alpha=0.8,
            label="Cluster mean"
        )

        plt.scatter(
            cluster_proto["global_prevalence"],
            cluster_proto["feature"],
            marker="x",
            s=70,
            label="Global mean"
        )

        plt.xlabel("Feature mean value")
        plt.ylabel("Feature")
        plt.title(f"CLAMP-style prototype - Cluster {cluster_id}")
        plt.legend()
        plt.tight_layout()
        plt.show()


def plot_clamp_heatmap(prototypes_df, top_n_features=30):
    selected_features = (
        prototypes_df
        .groupby("feature")["abs_prevalence_diff"]
        .max()
        .sort_values(ascending=False)
        .head(top_n_features)
        .index
    )

    heatmap_df = prototypes_df[
        prototypes_df["feature"].isin(selected_features)
    ].pivot_table(
        index="feature",
        columns="cluster",
        values="prevalence_diff",
        aggfunc="mean"
    ).fillna(0)

    plt.figure(figsize=(12, max(6, 0.35 * len(heatmap_df))))
    sns.heatmap(
        heatmap_df,
        cmap="coolwarm",
        center=0,
        annot=True,
        fmt=".2f"
    )

    plt.title("CLAMP-style feature differences by cluster")
    plt.xlabel("Cluster")
    plt.ylabel("Feature")
    plt.tight_layout()
    plt.show()

    return heatmap_df

In [ ]:
# ============================================================
# CELL 10 - Decision tree explanation of clusters
# ============================================================

def train_decision_tree_cluster_explainer(
    features_df,
    labels,
    max_depth=4,
    min_samples_leaf=10,
    random_state=123
):
    X = features_df.copy()
    y = np.asarray(labels)

    tree = DecisionTreeClassifier(
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        random_state=random_state,
        class_weight="balanced"
    )

    tree.fit(X, y)

    y_pred = tree.predict(X)

    metrics = {
        "tree_fidelity_accuracy": accuracy_score(y, y_pred),
        "tree_fidelity_balanced_accuracy": balanced_accuracy_score(y, y_pred),
        "max_depth": max_depth,
        "min_samples_leaf": min_samples_leaf,
        "n_leaves": tree.get_n_leaves(),
        "tree_depth": tree.get_depth(),
    }

    report = classification_report(
        y,
        y_pred,
        output_dict=True,
        zero_division=0
    )

    rules_text = export_text(
        tree,
        feature_names=list(X.columns)
    )

    return tree, metrics, report, rules_text


def plot_decision_tree_cluster_explainer(
    tree,
    features_df,
    title="Decision tree explanation of clusters"
):
    plt.figure(figsize=(24, 12))

    plot_tree(
        tree,
        feature_names=list(features_df.columns),
        class_names=[str(c) for c in tree.classes_],
        filled=True,
        rounded=True,
        fontsize=8
    )

    plt.title(title)
    plt.tight_layout()
    plt.show()


def get_decision_tree_feature_importance(tree, features_df):
    importance = pd.Series(
        tree.feature_importances_,
        index=features_df.columns
    ).sort_values(ascending=False)

    return importance[importance > 0]


def plot_tree_feature_importance(tree_feature_importance, top_n=20):
    top_imp = tree_feature_importance.head(top_n).sort_values()

    plt.figure(figsize=(10, max(4, 0.35 * len(top_imp))))
    plt.barh(top_imp.index, top_imp.values)
    plt.xlabel("Decision tree feature importance")
    plt.ylabel("Feature")
    plt.title("Most important features for explaining clusters")
    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# CELL 11 - Compare clusters with circle membership
# ============================================================

def compare_clusters_with_circles(membership_matrix, labels):
    df_cluster_labels = pd.DataFrame(
        {"cluster": labels},
        index=range(1, len(labels) + 1)
    )

    membership_clustered = membership_matrix.join(df_cluster_labels, how="inner")

    circle_counts_per_cluster = membership_clustered.groupby("cluster").sum()

    circle_props_per_cluster = circle_counts_per_cluster.div(
        circle_counts_per_cluster.sum(axis=1),
        axis=0
    )

    return circle_counts_per_cluster, circle_props_per_cluster


def plot_circle_composition(circle_props_per_cluster, title):
    circle_props_per_cluster.plot(
        kind="bar",
        stacked=True,
        figsize=(12, 6),
        colormap="tab20"
    )

    plt.title(title)
    plt.xlabel("Cluster")
    plt.ylabel("Proportion of Nodes after overlap grouping")
    plt.legend(title="Circle", bbox_to_anchor=(1.05, 1), loc="upper left")
    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# CELL 12 - Main clustering analysis function
# ============================================================

def run_clustering_analysis(
    embeddings,
    features_df,
    membership_matrix,
    n_clusters,
    output_folder,
    dataset_name,
    final_method="kmeans",
    random_state=123
):

    os.makedirs(output_folder, exist_ok=True)

    print("\n" + "=" * 80)
    print(f"FULL ANALYSIS FOR: {dataset_name}")
    print("=" * 80)

    results, metrics_df = compare_clustering_methods(
        embeddings=embeddings,
        n_clusters=n_clusters,
        random_state=random_state
    )

    print("\n=== Clustering metrics ===")
    print(metrics_df)

    metrics_df.to_csv(
        os.path.join(
            output_folder,
            f"{dataset_name}_metrics_comparison_k{n_clusters}.csv"
        )
    )

    final_result = results[final_method]

    if final_result["labels"] is None:
        raise RuntimeError(
            f"{dataset_name}: {final_method} could not fit. "
            f"Status: {final_result['metrics']['fit_status']}"
        )

    final_labels = final_result["labels"]
    effective_clusters = len(np.unique(final_labels))

    labels_df = save_cluster_labels(
        final_labels,
        os.path.join(
            output_folder,
            f"{dataset_name}_{final_method}_labels_k{n_clusters}.csv"
        )
    )

    cluster_counts = plot_cluster_sizes(final_labels)

    cluster_counts.to_csv(
        os.path.join(
            output_folder,
            f"{dataset_name}_cluster_sizes_k{n_clusters}.csv"
        )
    )

    status_df = pd.DataFrame([{
        "dataset": dataset_name,
        "requested_clusters": n_clusters,
        "effective_clusters": effective_clusters,
        "final_method": final_method,
        "analysis_status": (
            "valid"
            if effective_clusters >= 2
            else "collapsed_single_cluster"
        )
    }])

    status_df.to_csv(
        os.path.join(
            output_folder,
            f"{dataset_name}_analysis_status_k{n_clusters}.csv"
        ),
        index=False
    )

    if effective_clusters < 2:
        print("\n" + "!" * 80)
        print("CLUSTERING COLLAPSED TO ONE EFFECTIVE CLUSTER")
        print(
            "The embeddings do not support a meaningful multi-cluster "
            "interpretation in this configuration."
        )
        print("!" * 80)

        return {
            "status": "collapsed_single_cluster",
            "results": results,
            "metrics_df": metrics_df,
            "labels": final_labels,
            "labels_df": labels_df,
            "cluster_counts": cluster_counts,
            "effective_clusters": effective_clusters,
        }

    if final_method == "kmeans":
        centroides = results["kmeans"]["centroides"]

        centroides_df = pd.DataFrame(
            centroides,
            columns=[
                f"embedding_dim_{i}"
                for i in range(centroides.shape[1])
            ]
        )

        centroides_df.index.name = "cluster"

        print("\n=== KMeans centroids ===")
        print(centroides_df)

        centroides_df.to_csv(
            os.path.join(
                output_folder,
                f"{dataset_name}_kmeans_centroids_k{n_clusters}.csv"
            )
        )

    emb_2d = compute_tsne(
        embeddings,
        random_state=random_state
    )

    display_name = DISPLAY_NAMES.get(dataset_name, dataset_name)

    plot_tsne_clusters(
        emb_2d,
        final_labels,
        title=f"{display_name} - {final_method} - k={n_clusters}"
    )

    plot_feature_sparsity(features_df)

    cluster_feature_profiles = compute_cluster_feature_profiles(
        features_df,
        final_labels
    )

    print("\n=== Cluster feature profiles ===")
    print(cluster_feature_profiles)

    cluster_feature_profiles.to_csv(
        os.path.join(
            output_folder,
            f"{dataset_name}_cluster_feature_profiles_k{n_clusters}.csv"
        )
    )

    plot_feature_profile_heatmap(cluster_feature_profiles)

    sorted_features = print_features_above_threshold(
        cluster_feature_profiles,
        threshold=0.3
    )

    sorted_features.to_csv(
        os.path.join(
            output_folder,
            f"{dataset_name}_features_above_threshold_k{n_clusters}.csv"
        )
    )

    mi_scores = compute_mutual_information_features(
        features_df,
        final_labels
    )

    print("\n=== Top mutual information features ===")
    print(mi_scores.head(20))

    mi_scores.to_csv(
        os.path.join(
            output_folder,
            f"{dataset_name}_mutual_information_scores_k{n_clusters}.csv"
        )
    )

    plot_top_features_per_cluster(
        cluster_feature_profiles,
        threshold=0.4
    )

    clamp_prototypes = compute_clamp_feature_prototypes(
        features_df,
        final_labels,
        lower_q=0.10,
        upper_q=0.90,
        top_n=15
    )

    print("\n=== CLAMP prototypes ===")
    print(clamp_prototypes.head(30))

    clamp_prototypes.to_csv(
        os.path.join(
            output_folder,
            f"{dataset_name}_clamp_prototypes_k{n_clusters}.csv"
        ),
        index=False
    )

    plot_clamp_feature_prototypes(
        clamp_prototypes,
        top_n=10,
        value_col="cluster_prevalence"
    )

    clamp_heatmap = plot_clamp_heatmap(
        clamp_prototypes,
        top_n_features=30
    )

    clamp_heatmap.to_csv(
        os.path.join(
            output_folder,
            f"{dataset_name}_clamp_heatmap_k{n_clusters}.csv"
        )
    )

    tree, tree_metrics, tree_report, tree_rules = (
        train_decision_tree_cluster_explainer(
            features_df,
            final_labels,
            max_depth=4,
            min_samples_leaf=10,
            random_state=random_state
        )
    )

    print("\n=== Decision tree fidelity metrics ===")
    print(pd.DataFrame([tree_metrics]))

    print("\n=== Decision tree rules ===")
    print(tree_rules)

    plot_decision_tree_cluster_explainer(
        tree,
        features_df,
        title=f"{dataset_name} - Decision Tree"
    )

    tree_feature_importance = get_decision_tree_feature_importance(
        tree,
        features_df
    )

    print("\n=== Decision tree feature importance ===")
    print(tree_feature_importance.head(20))

    tree_feature_importance.to_csv(
        os.path.join(
            output_folder,
            f"{dataset_name}_tree_feature_importance_k{n_clusters}.csv"
        )
    )

    plot_tree_feature_importance(
        tree_feature_importance,
        top_n=20
    )

    circle_counts, circle_props = compare_clusters_with_circles(
        membership_matrix,
        final_labels
    )

    print("\n=== Circle proportions by cluster ===")
    print(circle_props)

    circle_counts.to_csv(
        os.path.join(
            output_folder,
            f"{dataset_name}_circle_counts_k{n_clusters}.csv"
        )
    )

    circle_props.to_csv(
        os.path.join(
            output_folder,
            f"{dataset_name}_circle_props_k{n_clusters}.csv"
        )
    )

    display_name = DISPLAY_NAMES.get(dataset_name, dataset_name)

    plot_circle_composition(
    circle_props,
    title=f"{display_name} - Circle Composition by Cluster"
)

    return {
        "status": "valid",
        "results": results,
        "metrics_df": metrics_df,
        "labels": final_labels,
        "labels_df": labels_df,
        "cluster_counts": cluster_counts,
        "effective_clusters": effective_clusters,
        "cluster_feature_profiles": cluster_feature_profiles,
        "sorted_features": sorted_features,
        "mi_scores": mi_scores,
        "clamp_prototypes": clamp_prototypes,
        "clamp_heatmap": clamp_heatmap,
        "tree": tree,
        "tree_metrics": tree_metrics,
        "tree_report": tree_report,
        "tree_rules": tree_rules,
        "tree_feature_importance": tree_feature_importance,
        "circle_counts": circle_counts,
        "circle_props": circle_props,
    }

In [ ]:
# ============================================================
# CELL 13 - Paths and global parameters
# ============================================================

dgi_folder = "facebook0_dgi"

features_path = "features_aligned_facebook0.csv"
featnames_path = "0.featnames"

ard_path = "ard_matrix_facebook0.csv"
membership_path = "membership_matrix_facebook0.csv"

# New folder: keeps previous Step 4 results untouched.
output_folder = "facebook0_clustering_training_embeddings_comparison"
os.makedirs(output_folder, exist_ok=True)

K_ALL_FEATURES = 8
K_CIRCLES_AND_STRUCTURE = 8

final_method = "kmeans"
random_state = 123

In [ ]:
# ============================================================
# CELL 14 - Load interpretation data
# ============================================================

features_all_df = load_features_for_synthetic_graphs(
    features_path=features_path,
    featnames_path=featnames_path
)

features_circles_df = load_ard_features(
    ard_path=ard_path
)

membership_matrix = load_membership_for_synthetic_order(
    membership_path=membership_path
)

membership_matrix = add_circle_overlap_column(
    membership_matrix
)

print("All features shape:", features_all_df.shape)
print("ARD circles-only features shape:", features_circles_df.shape)
print("Membership matrix shape:", membership_matrix.shape)

print("\nAll features preview:")
display(features_all_df.head())

print("\nARD features preview:")
display(features_circles_df.head())

print("\nMembership preview:")
display(membership_matrix.head())

In [ ]:
# ============================================================
# CELL 15 - Dataset registry
# ============================================================

dataset_registry = {
    "original_all_features": {
        "embedding_path": os.path.join(
            dgi_folder,
            "embeddings_original_all_features.npy"
        ),
        "features_df": features_all_df,
    },

    "original_circles_only": {
        "embedding_path": os.path.join(
            dgi_folder,
            "embeddings_original_circles_only.npy"
        ),
        "features_df": features_circles_df,
    },

    "original_graph_structure_identity_only": {
        "embedding_path": os.path.join(
            dgi_folder,
            "embeddings_original_graph_structure_identity_only.npy"
        ),
        # Identity was DGI input, but original profile features
        # are used only for post-hoc interpretation.
        "features_df": features_all_df,
    },

    "original_TRAINING_all_features": {
        "embedding_path": os.path.join(
            dgi_folder,
            "embeddings_original_TRAINING_all_features.npy"
        ),
        "features_df": features_all_df,
    },

    "original_TRAINING_circles_only": {
        "embedding_path": os.path.join(
            dgi_folder,
            "embeddings_original_TRAINING_circles_only.npy"
        ),
        "features_df": features_circles_df,
    },

    "original_TRAINING_graph_structure_identity_only": {
        "embedding_path": os.path.join(
            dgi_folder,
            "embeddings_original_TRAINING_"
            "graph_structure_identity_only.npy"
        ),
        # Identity was DGI input, but original profile features
        # are used only for post-hoc interpretation.
        "features_df": features_all_df,
    },

    "method1_independent_mean_all_features": {
        "embedding_path": os.path.join(
            dgi_folder,
            "embeddings_method1_independent_mean_all_features.npy"
        ),
        "features_df": features_all_df,
    },

    "method1_independent_mean_circles_only": {
        "embedding_path": os.path.join(
            dgi_folder,
            "embeddings_method1_independent_mean_circles_only.npy"
        ),
        "features_df": features_circles_df,
    },

    "method1_independent_mean_graph_structure_identity_only": {
        "embedding_path": os.path.join(
            dgi_folder,
            "embeddings_method1_independent_mean_"
            "graph_structure_identity_only.npy"
        ),
        "features_df": features_all_df,
    },

    "method2_shared_sequential_mean_all_features": {
        "embedding_path": os.path.join(
            dgi_folder,
            "embeddings_method2_shared_sequential_mean_all_features.npy"
        ),
        "features_df": features_all_df,
    },

    "method2_shared_sequential_mean_circles_only": {
        "embedding_path": os.path.join(
            dgi_folder,
            "embeddings_method2_shared_sequential_mean_circles_only.npy"
        ),
        "features_df": features_circles_df,
    },

    "method2_shared_sequential_mean_graph_structure_identity_only": {
        "embedding_path": os.path.join(
            dgi_folder,
            "embeddings_method2_shared_sequential_mean_"
            "graph_structure_identity_only.npy"
        ),
        "features_df": features_all_df,
    },

    "method3_batch_mean_all_features": {
        "embedding_path": os.path.join(
            dgi_folder,
            "embeddings_method3_batch_mean_all_features.npy"
        ),
        "features_df": features_all_df,
    },

    "method3_batch_mean_circles_only": {
        "embedding_path": os.path.join(
            dgi_folder,
            "embeddings_method3_batch_mean_circles_only.npy"
        ),
        "features_df": features_circles_df,
    },

    "method3_batch_mean_graph_structure_identity_only": {
        "embedding_path": os.path.join(
            dgi_folder,
            "embeddings_method3_batch_mean_"
            "graph_structure_identity_only.npy"
        ),
        "features_df": features_all_df,
    },


    "method2_shared_sequential_TRAINING_MEAN_all_features": {
        "embedding_path": os.path.join(
            dgi_folder,
            "embeddings_method2_shared_sequential_TRAINING_MEAN_all_features.npy"
        ),
        "features_df": features_all_df,
    },

    "method2_shared_sequential_TRAINING_MEAN_circles_only": {
        "embedding_path": os.path.join(
            dgi_folder,
            "embeddings_method2_shared_sequential_TRAINING_MEAN_circles_only.npy"
        ),
        "features_df": features_circles_df,
    },

    "method2_shared_sequential_TRAINING_MEAN_graph_structure_identity_only": {
        "embedding_path": os.path.join(
            dgi_folder,
            "embeddings_method2_shared_sequential_TRAINING_MEAN_"
            "graph_structure_identity_only.npy"
        ),
        "features_df": features_all_df,
    },

    "method3_batch_TRAINING_MEAN_all_features": {
        "embedding_path": os.path.join(
            dgi_folder,
            "embeddings_method3_batch_TRAINING_MEAN_all_features.npy"
        ),
        "features_df": features_all_df,
    },

    "method3_batch_TRAINING_MEAN_circles_only": {
        "embedding_path": os.path.join(
            dgi_folder,
            "embeddings_method3_batch_TRAINING_MEAN_circles_only.npy"
        ),
        "features_df": features_circles_df,
    },

    "method3_batch_TRAINING_MEAN_graph_structure_identity_only": {
        "embedding_path": os.path.join(
            dgi_folder,
            "embeddings_method3_batch_TRAINING_MEAN_"
            "graph_structure_identity_only.npy"
        ),
        "features_df": features_all_df,
    },
}

In [ ]:
# ============================================================
# CELL 16 - Helper to load and check one dataset
# ============================================================

def load_dataset_from_registry(dataset_name):
    dataset_info = dataset_registry[dataset_name]

    embedding_path = dataset_info["embedding_path"]
    features_df = dataset_info["features_df"]

    if not os.path.exists(embedding_path):
        raise FileNotFoundError(f"Missing embedding file: {embedding_path}")

    embeddings = load_embeddings(embedding_path)

    assert embeddings.shape[0] == features_df.shape[0], \
        f"{dataset_name}: embeddings/features mismatch"

    assert embeddings.shape[0] == membership_matrix.shape[0], \
        f"{dataset_name}: embeddings/membership mismatch"

    print(f"Dataset loaded: {dataset_name}")
    print("Embeddings shape:", embeddings.shape)
    print("Features shape:", features_df.shape)
    print("Membership shape:", membership_matrix.shape)

    return embeddings, features_df

In [ ]:
# ============================================================
# CELL 17 - Helper to run diagnostics for one dataset
# ============================================================

def run_diagnostics_for_dataset(dataset_name, embeddings):
    print(f"\nRunning diagnostics for: {dataset_name}")

    diagnostics_df = compute_clustering_diagnostics(
        embeddings,
        k_min=2,
        k_max=8,
        random_state=123
    )

    diagnostics_df["dataset"] = dataset_name

    diagnostics_df.to_csv(
        os.path.join(
            output_folder,
            f"{dataset_name}_all_diagnostics_k2_k8.csv"
        ),
        index=False
    )

    split_diagnostics_by_method(
        diagnostics_df=diagnostics_df,
        output_folder=output_folder,
        dataset_name=dataset_name
    )

    return diagnostics_df

In [ ]:
# ============================================================
# CELL 18 - Helper to run one complete dataset analysis
# ============================================================

def run_final_dataset_analysis(
    dataset_name,
    n_clusters,
    final_method="kmeans",
    random_state=123
):
    embeddings, features_df = load_dataset_from_registry(
        dataset_name
    )

    diagnostics = run_diagnostics_for_dataset(
        dataset_name,
        embeddings
    )

    result = run_clustering_analysis(
        embeddings=embeddings,
        features_df=features_df,
        membership_matrix=membership_matrix,
        n_clusters=n_clusters,
        output_folder=output_folder,
        dataset_name=dataset_name,
        final_method=final_method,
        random_state=random_state
    )

    return diagnostics, result

In [ ]:
# ============================================================
# CELL 19 - MODEL 0A - Original graph with all features
# ============================================================

diagnostics_original_all, result_original_all = (
    run_final_dataset_analysis(
        dataset_name="original_all_features",
        n_clusters=K_ALL_FEATURES,
        final_method=final_method,
        random_state=random_state
    )
)

In [ ]:
# ============================================================
# CELL 20 - MODEL 0B - Original graph with circles-only ARD
# ============================================================

diagnostics_original_circles, result_original_circles = (
    run_final_dataset_analysis(
        dataset_name="original_circles_only",
        n_clusters=K_CIRCLES_AND_STRUCTURE,
        final_method=final_method,
        random_state=random_state
    )
)

In [ ]:
# ============================================================
# CELL 21 - MODEL 0C - Original graph with structure only
# ============================================================

diagnostics_original_structure, result_original_structure = (
    run_final_dataset_analysis(
        dataset_name="original_graph_structure_identity_only",
        n_clusters=K_CIRCLES_AND_STRUCTURE,
        final_method=final_method,
        random_state=random_state
    )
)

In [ ]:
# ============================================================
# TRAINING-TIME MODEL 0A - Original graph with all features
# ============================================================

diagnostics_original_training_all, result_original_training_all = (
    run_final_dataset_analysis(
        dataset_name="original_TRAINING_all_features",
        n_clusters=K_ALL_FEATURES,
        final_method=final_method,
        random_state=random_state
    )
)


In [ ]:
# ============================================================
# TRAINING-TIME MODEL 0B - Original graph with circles-only ARD
# ============================================================

diagnostics_original_training_circles, result_original_training_circles = (
    run_final_dataset_analysis(
        dataset_name="original_TRAINING_circles_only",
        n_clusters=K_CIRCLES_AND_STRUCTURE,
        final_method=final_method,
        random_state=random_state
    )
)


In [ ]:
# ============================================================
# TRAINING-TIME MODEL 0C - Original graph with structure only
# ============================================================

diagnostics_original_training_structure, result_original_training_structure = (
    run_final_dataset_analysis(
        dataset_name="original_TRAINING_graph_structure_identity_only",
        n_clusters=K_CIRCLES_AND_STRUCTURE,
        final_method=final_method,
        random_state=random_state
    )
)


In [ ]:
# ============================================================
# CELL 22 - MODEL 1A - Independent DGI with all features
# ============================================================

diagnostics_method1_all, result_method1_all = (
    run_final_dataset_analysis(
        dataset_name="method1_independent_mean_all_features",
        n_clusters=K_ALL_FEATURES,
        final_method=final_method,
        random_state=random_state
    )
)

In [ ]:
# ============================================================
# CELL 23 - MODEL 1B - Independent DGI with circles-only ARD
# ============================================================

diagnostics_method1_circles, result_method1_circles = (
    run_final_dataset_analysis(
        dataset_name="method1_independent_mean_circles_only",
        n_clusters=K_CIRCLES_AND_STRUCTURE,
        final_method=final_method,
        random_state=random_state
    )
)

In [ ]:
# ============================================================
# CELL 24 - MODEL 1C - Independent DGI with structure only
# ============================================================

diagnostics_method1_structure, result_method1_structure = (
    run_final_dataset_analysis(
        dataset_name=(
            "method1_independent_mean_"
            "graph_structure_identity_only"
        ),
        n_clusters=K_CIRCLES_AND_STRUCTURE,
        final_method=final_method,
        random_state=random_state
    )
)

In [ ]:
# ============================================================
# CELL 25 - MODEL 2A - Shared sequential DGI with all features
# ============================================================

diagnostics_method2_all, result_method2_all = (
    run_final_dataset_analysis(
        dataset_name="method2_shared_sequential_mean_all_features",
        n_clusters=K_ALL_FEATURES,
        final_method=final_method,
        random_state=random_state
    )
)

In [ ]:
# ============================================================
# CELL 26 - MODEL 2B - Shared sequential DGI with circles-only ARD
# ============================================================

diagnostics_method2_circles, result_method2_circles = (
    run_final_dataset_analysis(
        dataset_name="method2_shared_sequential_mean_circles_only",
        n_clusters=K_CIRCLES_AND_STRUCTURE,
        final_method=final_method,
        random_state=random_state
    )
)

In [ ]:
# ============================================================
# CELL 27 - MODEL 2C - Shared sequential DGI with structure only
# ============================================================

diagnostics_method2_structure, result_method2_structure = (
    run_final_dataset_analysis(
        dataset_name=(
            "method2_shared_sequential_mean_"
            "graph_structure_identity_only"
        ),
        n_clusters=K_CIRCLES_AND_STRUCTURE,
        final_method=final_method,
        random_state=random_state
    )
)

In [ ]:
# ============================================================
# CELL 28 - MODEL 3A - Batch DGI with all features
# ============================================================

diagnostics_method3_all, result_method3_all = (
    run_final_dataset_analysis(
        dataset_name="method3_batch_mean_all_features",
        n_clusters=K_ALL_FEATURES,
        final_method=final_method,
        random_state=random_state
    )
)

In [ ]:
# ============================================================
# CELL 29 - MODEL 3B - Batch DGI with circles-only ARD
# ============================================================

diagnostics_method3_circles, result_method3_circles = (
    run_final_dataset_analysis(
        dataset_name="method3_batch_mean_circles_only",
        n_clusters=K_CIRCLES_AND_STRUCTURE,
        final_method=final_method,
        random_state=random_state
    )
)

In [ ]:
# ============================================================
# CELL 30 - MODEL 3C - Batch DGI with structure only
# ============================================================

diagnostics_method3_structure, result_method3_structure = (
    run_final_dataset_analysis(
        dataset_name=(
            "method3_batch_mean_"
            "graph_structure_identity_only"
        ),
        n_clusters=K_CIRCLES_AND_STRUCTURE,
        final_method=final_method,
        random_state=random_state
    )
)

In [ ]:
# ============================================================
# TRAINING-TIME MODEL 2A - One Encoder with all features
# ============================================================

diagnostics_method2_training_all, result_method2_training_all = (
    run_final_dataset_analysis(
        dataset_name="method2_shared_sequential_TRAINING_MEAN_all_features",
        n_clusters=K_ALL_FEATURES,
        final_method=final_method,
        random_state=random_state
    )
)


In [ ]:
# ============================================================
# TRAINING-TIME MODEL 2B - One Encoder with circles-only ARD
# ============================================================

diagnostics_method2_training_circles, result_method2_training_circles = (
    run_final_dataset_analysis(
        dataset_name="method2_shared_sequential_TRAINING_MEAN_circles_only",
        n_clusters=K_CIRCLES_AND_STRUCTURE,
        final_method=final_method,
        random_state=random_state
    )
)


In [ ]:
# ============================================================
# TRAINING-TIME MODEL 2C - One Encoder with structure only
# ============================================================

diagnostics_method2_training_structure, result_method2_training_structure = (
    run_final_dataset_analysis(
        dataset_name=(
            "method2_shared_sequential_TRAINING_MEAN_"
            "graph_structure_identity_only"
        ),
        n_clusters=K_CIRCLES_AND_STRUCTURE,
        final_method=final_method,
        random_state=random_state
    )
)


In [ ]:
# ============================================================
# TRAINING-TIME MODEL 3A - Batch with all features
# ============================================================

diagnostics_method3_training_all, result_method3_training_all = (
    run_final_dataset_analysis(
        dataset_name="method3_batch_TRAINING_MEAN_all_features",
        n_clusters=K_ALL_FEATURES,
        final_method=final_method,
        random_state=random_state
    )
)


In [ ]:
# ============================================================
# TRAINING-TIME MODEL 3B - Batch with circles-only ARD
# ============================================================

diagnostics_method3_training_circles, result_method3_training_circles = (
    run_final_dataset_analysis(
        dataset_name="method3_batch_TRAINING_MEAN_circles_only",
        n_clusters=K_CIRCLES_AND_STRUCTURE,
        final_method=final_method,
        random_state=random_state
    )
)


In [ ]:
# ============================================================
# TRAINING-TIME MODEL 3C - Batch with structure only
# ============================================================

diagnostics_method3_training_structure, result_method3_training_structure = (
    run_final_dataset_analysis(
        dataset_name=(
            "method3_batch_TRAINING_MEAN_"
            "graph_structure_identity_only"
        ),
        n_clusters=K_CIRCLES_AND_STRUCTURE,
        final_method=final_method,
        random_state=random_state
    )
)


In [ ]:
# ============================================================
# FINAL COMPARISON
# Post-training vs training-time embeddings
# ============================================================

comparison_pairs = [
    (
        "Original Graph",
        "all_features",
        result_original_all,
        result_original_training_all
    ),
    (
        "Original Graph",
        "circles_only",
        result_original_circles,
        result_original_training_circles
    ),
    (
        "Original Graph",
        "graph_structure_identity_only",
        result_original_structure,
        result_original_training_structure
    ),
    (
        "One Encoder",
        "all_features",
        result_method2_all,
        result_method2_training_all
    ),
    (
        "One Encoder",
        "circles_only",
        result_method2_circles,
        result_method2_training_circles
    ),
    (
        "One Encoder",
        "graph_structure_identity_only",
        result_method2_structure,
        result_method2_training_structure
    ),
    (
        "Batch",
        "all_features",
        result_method3_all,
        result_method3_training_all
    ),
    (
        "Batch",
        "circles_only",
        result_method3_circles,
        result_method3_training_circles
    ),
    (
        "Batch",
        "graph_structure_identity_only",
        result_method3_structure,
        result_method3_training_structure
    ),
]

comparison_rows = []

for model_name, feature_set, post_result, training_result in comparison_pairs:
    post_metrics = post_result["results"][final_method]["metrics"]
    training_metrics = training_result["results"][final_method]["metrics"]

    comparison_rows.append({
        "model": model_name,
        "feature_set": feature_set,
        "embedding_source": "post_training_reencoded",
        "silhouette": post_metrics["silhouette"],
        "calinski_harabasz": post_metrics["calinski_harabasz"],
        "davies_bouldin": post_metrics["davies_bouldin"],
        "n_clusters": post_metrics["n_clusters"],
        "fit_status": post_metrics["fit_status"],
    })

    comparison_rows.append({
        "model": model_name,
        "feature_set": feature_set,
        "embedding_source": "training_time",
        "silhouette": training_metrics["silhouette"],
        "calinski_harabasz": training_metrics["calinski_harabasz"],
        "davies_bouldin": training_metrics["davies_bouldin"],
        "n_clusters": training_metrics["n_clusters"],
        "fit_status": training_metrics["fit_status"],
    })

training_vs_post_comparison = pd.DataFrame(comparison_rows)

print("\n" + "=" * 90)
print("POST-TRAINING VS TRAINING-TIME EMBEDDING COMPARISON")
print("=" * 90)
print(training_vs_post_comparison)

training_vs_post_comparison.to_csv(
    os.path.join(
        output_folder,
        "training_vs_post_training_clustering_comparison.csv"
    ),
    index=False
)

print(
    "\nSaved comparison table to:",
    os.path.join(
        output_folder,
        "training_vs_post_training_clustering_comparison.csv"
    )
)
